# Test Forecasting Models: Comparación con Frontend

Este notebook prueba los 4 modelos de forecasting disponibles en DashAI:
1. **ARIMA**
2. **SARIMAX**
3. **Prophet**
4. **SklearnMultiStepForecaster** (MultiOutputRegressor con sklearn)

El objetivo es verificar que las métricas de evaluación calculadas aquí coincidan con las del frontend de DashAI.

## 1. Import Required Libraries

In [ ]:
import sys
import warnings

warnings.filterwarnings("ignore")

# Add DashAI to path
sys.path.insert(0, "/home/ivan/projects/ProyectoTitulo/DashAI")

# DashAI imports
import matplotlib.pyplot as plt
import numpy as np

# Standard libraries
import pandas as pd
from datasets import Dataset

from DashAI.back.dataloaders.classes.dashai_dataset import (
    to_dashai_dataset,
)
from DashAI.back.metrics.forecasting.mape import MAPE
from DashAI.back.metrics.regression.mae import MAE
from DashAI.back.metrics.regression.rmse import RMSE
from DashAI.back.models.forecasting.arima_model import ARIMAModel
from DashAI.back.models.forecasting.prophet_model import ProphetModel
from DashAI.back.models.forecasting.sarimax_model import SARIMAXModel
from DashAI.back.models.forecasting.sklearn_multistep_forecaster import (
    SklearnMultiStepForecaster,
)

print("✅ Imports completados")

## 2. Load and Prepare Dataset

Cargaremos un dataset de series temporales (ejemplo: Wikipedia Page Views o Airline Passengers)

In [ ]:
# Cargar dataset de ejemplo (Wikipedia Page Views - Prophet example data)
# Puedes usar tu propio dataset cambiando esta sección

# Opción 1: Usar dataset de Prophet

# Generar datos sintéticos de ejemplo (similar al dataset del frontend)
np.random.seed(42)
dates = pd.date_range(start="2015-01-01", end="2023-12-31", freq="D")
trend = np.linspace(10, 20, len(dates))
seasonality = 5 * np.sin(2 * np.pi * np.arange(len(dates)) / 365.25)
noise = np.random.normal(0, 1, len(dates))
values = trend + seasonality + noise

df = pd.DataFrame({"ds": dates, "y": values})

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['ds'].min()} to {df['ds'].max()}")
print("\nFirst rows:")
print(df.head())
print("\nDataset info:")
print(df.info())

## 3. Configure Temporal Splitter

Usaremos splits temporales como lo hace el frontend de DashAI:
- **Train**: 70% de los datos más antiguos
- **Validation**: 15% siguiente
- **Test**: 15% más reciente

In [ ]:
# Convertir a DashAIDataset
hf_dataset = Dataset.from_pandas(df)
dashai_dataset = to_dashai_dataset(hf_dataset)

# Crear splits temporales (70/15/15)
n = len(df)
train_size = int(0.70 * n)
val_size = int(0.15 * n)

train_end = train_size
val_end = train_end + val_size

splits = {
    "train_indexes": list(range(train_end)),
    "val_indexes": list(range(train_end, val_end)),
    "test_indexes": list(range(val_end, n)),
}

print(f"Total samples: {n}")
print(
    f"Train: {len(splits['train_indexes'])} samples ({len(splits['train_indexes']) / n * 100:.1f}%)"
)
print(
    f"Validation: {len(splits['val_indexes'])} samples ({len(splits['val_indexes']) / n * 100:.1f}%)"
)
print(
    f"Test: {len(splits['test_indexes'])} samples ({len(splits['test_indexes']) / n * 100:.1f}%)"
)

# Crear subsets para cada split
train_df = df.iloc[splits["train_indexes"]].reset_index(drop=True)
val_df = df.iloc[splits["val_indexes"]].reset_index(drop=True)
test_df = df.iloc[splits["test_indexes"]].reset_index(drop=True)

print(f"\nTrain date range: {train_df['ds'].min()} to {train_df['ds'].max()}")
print(f"Val date range: {val_df['ds'].min()} to {val_df['ds'].max()}")
print(f"Test date range: {test_df['ds'].min()} to {test_df['ds'].max()}")

## 4. Prepare Data for Models

Preparamos los datasets x (inputs) e y (outputs) como lo hace DashAI internamente

In [ ]:
# Crear datasets separados para x e y como DashAI
x_train = to_dashai_dataset(Dataset.from_pandas(train_df[["ds"]]))
y_train = to_dashai_dataset(Dataset.from_pandas(train_df[["y"]]))

x_val = to_dashai_dataset(Dataset.from_pandas(val_df[["ds"]]))
y_val = to_dashai_dataset(Dataset.from_pandas(val_df[["y"]]))

x_test = to_dashai_dataset(Dataset.from_pandas(test_df[["ds"]]))
y_test = to_dashai_dataset(Dataset.from_pandas(test_df[["y"]]))

# Metadata temporal (como la genera ForecastingTask)
temporal_metadata = {
    "timestamp_col": "ds",
    "target_col": "y",
    "exog_cols": [],
    "frequency": "D",
    "start_date": df["ds"].min(),
    "end_date": df["ds"].max(),
    "n_periods": len(df),
}

print("✅ Datasets preparados para entrenamiento")
print(f"X_train shape: {x_train.to_pandas().shape}")
print(f"Y_train shape: {y_train.to_pandas().shape}")

## 5. Model 1: ARIMA

Entrenaremos el modelo ARIMA con parámetros por defecto

In [ ]:
print("=" * 60)
print("MODELO 1: ARIMA")
print("=" * 60)

# Crear y entrenar modelo ARIMA
arima_model = ARIMAModel(
    order_p=1,
    order_d=1,
    order_q=1,
    trend="n",  # Sin tendencia (fix aplicado)
)

# Entrenar
print("\n🔧 Entrenando ARIMA...")
arima_model.fit(x_train, y_train, temporal_metadata=temporal_metadata)
print("✅ ARIMA entrenado")

# Predicciones en cada split
print("\n📊 Generando predicciones...")
arima_pred_train = arima_model.predict(x_pred=x_train)
arima_pred_val = arima_model.predict(x_pred=x_val)
arima_pred_test = arima_model.predict(x_pred=x_test)

print(f"Train predictions shape: {arima_pred_train.shape}")
print(f"Val predictions shape: {arima_pred_val.shape}")
print(f"Test predictions shape: {arima_pred_test.shape}")

## 6. Model 2: SARIMAX

Entrenaremos el modelo SARIMAX con componentes estacionales

In [ ]:
print("=" * 60)
print("MODELO 2: SARIMAX")
print("=" * 60)

# Crear y entrenar modelo SARIMAX
sarimax_model = SARIMAXModel(
    order_p=1,
    order_d=1,
    order_q=1,
    seasonal_order_P=1,
    seasonal_order_D=1,
    seasonal_order_Q=1,
    seasonal_period=7,  # Estacionalidad semanal
    trend="n",
)

# Entrenar
print("\n🔧 Entrenando SARIMAX...")
sarimax_model.fit(x_train, y_train, temporal_metadata=temporal_metadata)
print("✅ SARIMAX entrenado")

# Predicciones
print("\n📊 Generando predicciones...")
sarimax_pred_train = sarimax_model.predict(x_pred=x_train)
sarimax_pred_val = sarimax_model.predict(x_pred=x_val)
sarimax_pred_test = sarimax_model.predict(x_pred=x_test)

print(f"Train predictions shape: {sarimax_pred_train.shape}")
print(f"Val predictions shape: {sarimax_pred_val.shape}")
print(f"Test predictions shape: {sarimax_pred_test.shape}")

## 7. Model 3: Prophet

Entrenaremos el modelo Prophet de Facebook

In [ ]:
print("=" * 60)
print("MODELO 3: PROPHET")
print("=" * 60)

# Crear y entrenar modelo Prophet
prophet_model = ProphetModel(
    seasonality_mode="additive",
    yearly_seasonality="auto",
    weekly_seasonality="auto",
    daily_seasonality="auto",
)

# Entrenar
print("\n🔧 Entrenando Prophet...")
prophet_model.fit(x_train, y_train, temporal_metadata=temporal_metadata)
print("✅ Prophet entrenado")

# Predicciones
print("\n📊 Generando predicciones...")
prophet_pred_train = prophet_model.predict(x_pred=x_train)
prophet_pred_val = prophet_model.predict(x_pred=x_val)
prophet_pred_test = prophet_model.predict(x_pred=x_test)

print(f"Train predictions shape: {prophet_pred_train.shape}")
print(f"Val predictions shape: {prophet_pred_val.shape}")
print(f"Test predictions shape: {prophet_pred_test.shape}")

## 8. Model 4: SklearnMultiStepForecaster

Entrenaremos el modelo basado en sklearn con lag features automáticos

In [ ]:
print("=" * 60)
print("MODELO 4: SKLEARN MULTISTEP FORECASTER")
print("=" * 60)

# Crear y entrenar modelo SklearnMultiStepForecaster
sklearn_model = SklearnMultiStepForecaster(
    base_estimator="linear", window_size=7, forecast_strategy="direct"
)

# Entrenar
print("\n🔧 Entrenando SklearnMultiStepForecaster...")
sklearn_model.fit(x_train, y_train, temporal_metadata=temporal_metadata)
print("✅ SklearnMultiStepForecaster entrenado")

# Predicciones
print("\n📊 Generando predicciones...")
sklearn_pred_train = sklearn_model.predict(x_pred=x_train)
sklearn_pred_val = sklearn_model.predict(x_pred=x_val)
sklearn_pred_test = sklearn_model.predict(x_pred=x_test)

print(f"Train predictions shape: {sklearn_pred_train.shape}")
print(f"Val predictions shape: {sklearn_pred_val.shape}")
print(f"Test predictions shape: {sklearn_pred_test.shape}")

# Nota sobre NaN
print(
    "\n⚠️  Nota: Los primeros 'window_size' valores pueden ser NaN (sin suficientes lags)"
)

## 9. Calculate Evaluation Metrics

Calcularemos las métricas usando exactamente la misma lógica que DashAI (MAE, RMSE, MAPE)

In [ ]:
# Instanciar métricas
mae_metric = MAE()
rmse_metric = RMSE()
mape_metric = MAPE()


def calculate_metrics(y_true, y_pred, model_name, split_name):
    """Calcular métricas como lo hace DashAI"""
    try:
        mae = mae_metric.run(y_true, y_pred)
        rmse = rmse_metric.run(y_true, y_pred)
        mape = mape_metric.run(y_true, y_pred)

        return {
            "model": model_name,
            "split": split_name,
            "MAE": mae,
            "RMSE": rmse,
            "MAPE": mape,
        }
    except Exception as e:
        print(f"⚠️  Error calculando métricas para {model_name} ({split_name}): {e}")
        return {
            "model": model_name,
            "split": split_name,
            "MAE": None,
            "RMSE": None,
            "MAPE": None,
        }


# Calcular métricas para todos los modelos
results = []

# ARIMA
results.append(calculate_metrics(y_train, arima_pred_train, "ARIMA", "train"))
results.append(calculate_metrics(y_val, arima_pred_val, "ARIMA", "validation"))
results.append(calculate_metrics(y_test, arima_pred_test, "ARIMA", "test"))

# SARIMAX
results.append(calculate_metrics(y_train, sarimax_pred_train, "SARIMAX", "train"))
results.append(calculate_metrics(y_val, sarimax_pred_val, "SARIMAX", "validation"))
results.append(calculate_metrics(y_test, sarimax_pred_test, "SARIMAX", "test"))

# Prophet
results.append(calculate_metrics(y_train, prophet_pred_train, "Prophet", "train"))
results.append(calculate_metrics(y_val, prophet_pred_val, "Prophet", "validation"))
results.append(calculate_metrics(y_test, prophet_pred_test, "Prophet", "test"))

# Sklearn
results.append(
    calculate_metrics(y_train, sklearn_pred_train, "SklearnMultiStep", "train")
)
results.append(
    calculate_metrics(y_val, sklearn_pred_val, "SklearnMultiStep", "validation")
)
results.append(calculate_metrics(y_test, sklearn_pred_test, "SklearnMultiStep", "test"))

print("✅ Métricas calculadas para todos los modelos")

## 10. Compare Results

Tabla comparativa de métricas para todos los modelos

In [ ]:
# Crear DataFrame con resultados
results_df = pd.DataFrame(results)

# Mostrar tabla completa
print("\n" + "=" * 80)
print("RESULTADOS DE MÉTRICAS - TODOS LOS MODELOS")
print("=" * 80)
print(results_df.to_string(index=False))

# Tabla pivoteada para mejor visualización
print("\n" + "=" * 80)
print("COMPARACIÓN POR MODELO Y SPLIT")
print("=" * 80)

for metric in ["MAE", "RMSE", "MAPE"]:
    pivot = results_df.pivot(index="model", columns="split", values=metric)
    print(f"\n📊 {metric}:")
    print(pivot.to_string())

# Identificar el mejor modelo por split (menor MAE)
print("\n" + "=" * 80)
print("MEJOR MODELO POR SPLIT (según MAE)")
print("=" * 80)
for split in ["train", "validation", "test"]:
    split_data = results_df[results_df["split"] == split]
    best_model = split_data.loc[split_data["MAE"].idxmin()]
    print(f"\n{split.upper()}: {best_model['model']}")
    print(f"  MAE: {best_model['MAE']:.4f}")
    print(f"  RMSE: {best_model['RMSE']:.4f}")
    print(f"  MAPE: {best_model['MAPE']:.4f}")

## 11. Visualize Model Performance

Gráficos comparativos de predicciones vs valores reales

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(
    "Predicciones vs Valores Reales - Split de Test", fontsize=16, fontweight="bold"
)

models_data = [
    ("ARIMA", arima_pred_test),
    ("SARIMAX", sarimax_pred_test),
    ("Prophet", prophet_pred_test),
    ("SklearnMultiStep", sklearn_pred_test),
]

y_test_array = y_test.to_pandas()["y"].values

for idx, (model_name, predictions) in enumerate(models_data):
    ax = axes[idx // 2, idx % 2]

    # Filtrar NaN si existen
    mask = ~np.isnan(predictions)
    x_plot = np.arange(len(predictions))[mask]
    y_true_plot = y_test_array[mask]
    y_pred_plot = predictions[mask]

    ax.plot(x_plot, y_true_plot, label="Real", color="blue", alpha=0.7, linewidth=2)
    ax.plot(
        x_plot,
        y_pred_plot,
        label="Predicción",
        color="red",
        alpha=0.7,
        linewidth=2,
        linestyle="--",
    )

    ax.set_title(f"{model_name}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Time Index", fontsize=12)
    ax.set_ylabel("Value", fontsize=12)
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Visualizaciones generadas")

## 12. Compare with Frontend Results

Para verificar que las métricas coinciden con el frontend:

1. **En el frontend de DashAI**, entrena cada modelo con el mismo dataset y parámetros
2. **Compara las métricas** de la tabla anterior con las del frontend
3. **Verifica** que los valores sean idénticos (o muy cercanos debido a redondeo)

### Checklist de Verificación:
- [ ] ARIMA: MAE, RMSE, MAPE coinciden
- [ ] SARIMAX: MAE, RMSE, MAPE coinciden  
- [ ] Prophet: MAE, RMSE, MAPE coinciden
- [ ] SklearnMultiStepForecaster: MAE, RMSE, MAPE coinciden

**Nota**: Pequeñas diferencias (<0.01%) pueden deberse a:
- Redondeo de punto flotante
- Versiones diferentes de librerías
- Inicialización aleatoria (si aplica)

## 13. Export Results for Comparison

Guardamos los resultados en un archivo CSV para fácil comparación

In [ ]:
# Guardar resultados en CSV
output_file = (
    "/home/ivan/projects/ProyectoTitulo/DashAI/forecasting_metrics_comparison.csv"
)
results_df.to_csv(output_file, index=False)
print(f"✅ Resultados guardados en: {output_file}")

# Mostrar resumen final
print("\n" + "=" * 80)
print("RESUMEN FINAL")
print("=" * 80)
print("\n📊 Total de modelos evaluados: 4")
print("📊 Splits evaluados: train, validation, test")
print("📊 Métricas calculadas: MAE, RMSE, MAPE")
print(f"\n💾 Archivo de resultados: {output_file}")
print(
    "\n🎯 Próximo paso: Comparar estos resultados con las métricas del frontend de DashAI"
)
print("   usando el mismo dataset y configuración de splits.")